In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings("ignore")

/home/alessandrom/anaconda3/envs/bias_correction_conda/lib/python3.10/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.31.0 or higher is recommended. You are running version 2.16.0
  warnings.warn(


In [7]:
def opening_downscaling_cmip(model, exp, country, month, start, end):
    downscaled_path = f'../data/downscaled_data/{country}/{exp}/{model}/'
    
    if exp == 'historical':
        start_year = 1985
        end_year   = 2014
    else:
        start_year = start
        end_year   = end
    ds = xr.open_dataset(downscaled_path+f'ssrd_downscaled_{country}_m{month}_{start_year}_{end_year}.nc')
    return ds

In [14]:
def calculate_future_delta(atlas_path, country, month, hist_ds, ssp_ds, exp, start, end):
    pres_ds   = xr.open_dataset(atlas_path+f'ssrd_integrated_{country}_m{month}.nc')
    
    delta     = (ssp_ds.ssrd_downscaled - hist_ds.ssrd_downscaled).to_dataset(name='delta')
    future_ds = (pres_ds.ssrd_integrated + delta.delta).to_dataset(name='ssrd_integrated')
    future_ds['delta'] = delta.delta

    output_path = atlas_path+f'{exp}/{model}/'
    output_dir = os.path.dirname(output_path)
    os.makedirs(output_dir, exist_ok=True)
    future_ds.to_netcdf(output_path+f'ssrd_{country}_m{month}_{exp}_{start}_{end}.nc')
    return future_ds

In [22]:
month = 1
country = 'croatia'
model = 'CNRM-ESM2-1'
start = 2030 #SELECT THE STARTING YEAR OF THE DECADE I WANT TO ANALYZE. e.g 2030, 2050, 2070
end   = 2049 #SELECT THE ENDING YEAR OF THE DECADE I WANT TO ANALYZE. e.g. 2049,2069,2089

atlas_path = f'../data/atlas_data/{country}/'

In [16]:
hist_ds = opening_downscaling_cmip(model, 'historical', country, month, start, end)

In [17]:
s370_ds = opening_downscaling_cmip(model, 'ssp370', country, month, start, end)
    
ssp370_proj = calculate_future_delta(atlas_path, country, month, hist_ds, s370_ds, 'ssp370', start, end)

In [18]:
s585_ds = opening_downscaling_cmip(model, 'ssp585', country, month, start, end)
    
ssp585_proj = calculate_future_delta(atlas_path, country, month, hist_ds, s585_ds, 'ssp585', start, end)